In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv("../data/processed/matches_clean.csv")
deliveries = pd.read_csv("../data/processed/deliveries_clean.csv")

print(matches.shape, deliveries.shape)

(594, 10) (138834, 12)


In [2]:
first = deliveries[deliveries["inning"] == 1].groupby("match_id")["total_runs"].sum().reset_index()
first["target"] = first["total_runs"] + 1

second = deliveries[deliveries["inning"] == 2].merge(first[["match_id", "target"]], on="match_id")
second = second.merge(matches[["id", "winner", "city"]], left_on="match_id", right_on="id")

print(second.shape)

(67817, 16)


In [3]:
second["current_score"] = second.groupby("match_id")["total_runs"].cumsum()
second["runs_left"] = second["target"] - second["current_score"]

second["balls_bowled"] = (second["over"] - 1) * 6 + second["ball"]
second["balls_left"] = 120 - second["balls_bowled"]

second["wickets_fallen"] = second.groupby("match_id")["is_wicket"].cumsum()
second["wickets_left"] = 10 - second["wickets_fallen"]

second["crr"] = second["current_score"] * 6 / second["balls_bowled"].clip(lower=1)
second["rrr"] = second["runs_left"] * 6 / second["balls_left"].clip(lower=1)

second["result"] = (second["batting_team"] == second["winner"]).astype(int)

final = second[(second["balls_left"] > 0) & (second["runs_left"] > 0)]
final = final[["match_id", "batting_team", "bowling_team", "city", "runs_left",
               "balls_left", "wickets_left", "target", "crr", "rrr", "result"]].dropna()
final = final.reset_index(drop=True)

print(final.shape)
final.head()

(67249, 11)


,match_id,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,target,crr,rrr,result
0,1,Mumbai Indians,Chennai Super Kings,Mumbai,217,119,10,218,6.0,10.941176,0
1,1,Mumbai Indians,Chennai Super Kings,Mumbai,217,118,10,218,3.0,11.033898,0
2,1,Mumbai Indians,Chennai Super Kings,Mumbai,213,117,10,218,10.0,10.923077,0
3,1,Mumbai Indians,Chennai Super Kings,Mumbai,209,116,10,218,13.5,10.810345,0
4,1,Mumbai Indians,Chennai Super Kings,Mumbai,208,115,10,218,12.0,10.852174,0


In [4]:
from sklearn.model_selection import GroupShuffleSplit

X = final.drop(columns=["result"])
y = final["result"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
train_idx, test_idx = next(splitter.split(X, y, groups=X["match_id"]))

X_train = X.iloc[train_idx].drop(columns=["match_id"])
X_test = X.iloc[test_idx].drop(columns=["match_id"])
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_train.shape, X_test.shape)

(53606, 9) (13643, 9)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def make_prep():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"),
          ["batting_team", "bowling_team", "city"])],
        remainder="passthrough",
    )

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=1),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", make_prep()), ("model", clf)])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = (acc, pipe)
    print(name, round(acc * 100, 1), "%")

c:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression 81.7 %
Random Forest 78.9 %


In [6]:
baseline_pred = (X_test["crr"] >= X_test["rrr"]).astype(int)
print("Baseline (ahead on run rate):", round(accuracy_score(y_test, baseline_pred) * 100, 1), "%")

Baseline (ahead on run rate): 70.7 %


In [7]:
import joblib

best_name = max(results, key=lambda k: results[k][0])
print("Best model:", best_name)

joblib.dump(results[best_name][1], "../models/win_predictor.pkl")
print("Saved!")

Best model: Logistic Regression
Saved!
